<a href="https://colab.research.google.com/github/yuki-gu/ai-colab-notebooks/blob/main/PyTorch_CheatSheet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PyTorch チートシート
PyTorchの[Learn the Basics](https://docs.pytorch.org/tutorials/beginner/basics/intro.html)の内容を元に、PyTorchの使い方をまとめたものです。

以下の用途でお使いいただけます。
- PyTorchを初めて使う時のQuick Startとして
- PyTorchを勉強するときの学習教材として
- PyTorchコードを書く時のCheatSheetとして

## PyTorchで学習を行うまで

In [ ]:
# @title 常にimport

import torch
from torch import nn
from torch.utils.data import DataLoader  # datasetsをforで回せるようにする

%matplotlib inline
import matplotlib.pyplot as plt

---

## GPUの利用

### 流れ
1. データをGPUに送る
1. モデルをGPUに送る
1. GPU上でモデルの学習を行う

In [ ]:
# @title GPUが使えるか確認
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using device: {device}")

Using device: cpu


In [ ]:
data = torch.tensor(1)

data.to(device)  # GPUに送る
data.to("cpu")  # CPUに送る

tensor(1)

---

## データ準備

### 基本クラス
torch.utils.data.Dataset : データセットのデータ(入力データ+教師ラベル)を格納するだけのクラス

torch.utils.data.DataLoader : Datasetをミニバッチ化し、forで回せるようにするクラス

<br>

### torch内データセットの利用
データの種類ごとにライブラリが存在 (データセット、データ処理関数、定義済みモデルが含まれる)
- 自然言語: [torchtext](https://pytorch.org/text/stable/datasets.html)
- 画像: [torchvision](https://pytorch.org/vision/stable/datasets.html)
- 音声: [torchaudio](https://pytorch.org/audio/stable/datasets.html)

In [ ]:
# @title torchvision内のMNISTデータセットを取得

from torchvision import datasets  # torch.utils.data.Datasetを継承したクラスを含む
from torchvision.transforms import v2  # 前処理用

training_data = datasets.MNIST(
    root="data",  # データを置くディレクトリ
    train=True,  # 学習用データを取得
    download=True,  # rootにデータが無いならダウンロードして持ってくる
    transform=v2.Compose([  # 入力データの前処理
        v2.ToImage(),  # PIL画像 -> Image(TVTensor)
        v2.ToDtype(torch.float32, scale=True)  # [0,1]に変換
    ]),
    target_transform=torch.tensor  # 教師ラベルの前処理
)

test_data = datasets.MNIST(
    root="data",
    train=False,  # テスト用データを取得
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)]),
    target_transform=torch.tensor
)

In [ ]:
# @title torch.utils.data.Dataset

print(f'len: {len(training_data)}')  # Datasetのデータ数

item0 = training_data[0]  # Datasetにインデックスでアクセス
print(f'type: {type(item0)}, len: {len(item0)}')  # tuple型
print(f'shape: ({item0[0].shape}, {item0[1].shape})')  # (入力データ, 教師ラベル)
print(item0)


# 独自Dataset
from torch.utils.data import Dataset
class CustomDataset(Dataset):
  def __init__(self, input_data_list, target_data_list, transform=None, target_transform=None):
    # ファイルからデータを読み込み
    self.input_data_list = input_data_list
    self.target_data_list = target_data_list
    self.transform = transform
    self.target_transform = target_transform

  def __len__(self):
    # データセットのデータ数を返す
    return len(self.target_data_list)

  def __getitem__(self, idx):
    # idx番目のデータを返す
    input_data = self.input_data_list[idx]
    target_data = self.target_data_list[idx]

    if self.transform:
        input_data = self.transform(input_data)
    if self.target_transform:
        target_data = self.target_transform(target_data)
    return input_data, target_data


len: 60000
type: <class 'tuple'>
shape: (torch.Size([1, 28, 28]), torch.Size([]))
(Image([[[0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.00

In [ ]:
# @title torch.utils.data.DataLoader

from torch.utils.data import DataLoader

batch_size = 64  # ハイパーパラメータ

# DataLoaderの作成: ミニバッチ化 + データシャッフル
train_dataloader = DataLoader(training_data, batch_size=batch_size, shuffle=True)
test_dataloader = DataLoader(test_data, batch_size=batch_size, shuffle=True)

# forで回せるようになる
for input_batch, target_batch in train_dataloader:
  print(input_batch.shape)
  print(target_batch.shape)
  break

torch.Size([64, 1, 28, 28])
torch.Size([64])


In [ ]:
# @title 前処理+データ拡張(torchvision.transforms.v2)
# 参照: https://pytorch.org/vision/stable/transforms.html

from torchvision.transforms import v2  # 前処理(transform)を行うクラスを含む

# 変換の定義
transforms = v2.Compose([
    v2.ToImage(),  # PIL画像 -> Image(TVTensor)
    v2.ToDtype(torch.uint8, scale=True),  # [0,255]に変換 (画像変換はuint8で行う)
    v2.CenterCrop(size=28),  # 中央を切り抜いて正方形に
    v2.Resize(size=(16, 16)),  # リサイズ
    v2.RandomVerticalFlip(), v2.RandomHorizontalFlip(),  # 50%で上下左右反転
    v2.RandomRotation((-90, 90)),  # ランダム回転 (-90～90°)
    v2.ToDtype(torch.float32, scale=True),  # [0,1]に変換 (Normalizeはfloatで行う)
])

# 変換の実行
img = training_data[0][0]
out = transforms(img)

# Datasetに適用
transformed_dataset = datasets.MNIST(
    root="data",
    train=True,
    download=True,
    transform=transforms,  # 入力データの前処理
    # target_transform=target_transforms  # 教師ラベルの前処理
)



# 画像とTVTensors(物体検出のBoundingBox、セグメンテーションのMask、動画)の同時変換
from torchvision import tv_tensors

boxes = tv_tensors.BoundingBoxes(  # BoundingBoxを表すクラス (Tensorのサブクラス)
    [
        [15, 10, 370, 510],
        [275, 340, 510, 510],
        [130, 345, 210, 425]
    ],
    format="XYXY", canvas_size=(28, 28))
# tv_tensors.Mask  # Maskを表すクラス
# tv_tensors.Video  # 動画を表すクラス

out_img, out_boxes = transforms(img, boxes)  # 画像とBoundingBoxを同時に変換 (transformsに第2引数を渡す)
out_img, out_boxes = transforms(img, {"boxes": boxes, "path": "path/to/file"})  # dictを使えば補足情報をつけられる

detection_dataset = datasets.CocoDetection(
    "images_dir", "annotation_file",
    transforms=transforms  # transformではない (BoundingBoxも同時変換)
)

# DatasetをTVTensorsに対応させる
detection_dataset = datasets.wrap_dataset_for_transforms_v2(detection_dataset)

---

## 学習するネットワークの作成

### [torch.nn](https://pytorch.org/docs/stable/nn.html) : ネットワークを構成するクラスを含むライブラリ
- Sequential : 複数のレイヤーを連結させるコンテナ
- レイヤー
  - Linear(入力数, 出力数) - 全結合層
  - Dropout(0.5) - 確率pで切断
  - Conv2d(入力数, 出力数, kernel_size=3, stride=1, padding=1)
  - MaxPool2d(kernel_size=2, stride=2, padding=0)
  - RNN
  - LSTM
  - Transformer
- 活性化関数
  - ReLU()
  - Sigmoid()
  - Tanh()
  - Softmax(dim=1)
- 損失関数
  - L1Loss()(y, y_hat) - 平均絶対誤差(MAE)
  - MSELoss()(y, y_hat) - 平均二乗誤差
  - CrossEntropyLoss()(y, y_hat)
    - CrossEntropyLoss = LogSoftmax + NLLLoss(CrossEntropy)
    - yはSoftmax関数に通す前のLinearの出力値
    - y_hatはクラス番号 or One-Hotベクトル
  - HuberLoss()(y, y_hat)

<br>

[torch.optim](https://pytorch.org/docs/stable/optim.html) : 最適化手法のクラスを含む
- SGD - 確率的勾配降下法 (+モーメンタム)
- Adam

<br>

### torch内データセットの利用
データ種類別ライブラリの中に定義済みモデルが存在
- 自然言語: [torchtext](https://pytorch.org/text/main/models.html)
- 画像: [torchvision](https://pytorch.org/vision/main/models.html)
- 音声: [torchaudio](https://pytorch.org/audio/main/models.html)

In [ ]:
# @title モデルの定義(クラスの作成)
from torch import nn

class NeuralNetwork(nn.Module):  # nn.Moduleのサブクラスとしてネットワークを定義
  def __init__(self):
    super().__init__()

    # レイヤーの定義
    self.flatten = nn.Flatten()
    self.linear_relu_stack = nn.Sequential(  # レイヤーをまとめる
        nn.Linear(28*28, 512),  # 全結合層 (入力サイズ, 出力サイズ)
        nn.ReLU(),  # ReLU活性化関数
        nn.Linear(512, 512),
        nn.ReLU(),
        nn.Linear(512, 10),
    )

  def forward(self, x):
    # 順伝播の処理
    x = self.flatten(x)
    logits = self.linear_relu_stack(x)
    return logits

In [ ]:
# @title モデルのインスタンスを作成
model = NeuralNetwork()
print(model)  # モデル構造
print(model.parameters())  # モデルのパラメータ

model = model.to(device)  # モデルをGPUに送る

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)
<generator object Module.parameters at 0x7abf25439700>


In [ ]:
# @title 順伝播
X = torch.rand(1, 1, 28, 28)  # 入力データ (batch_size, num_channels, height, width)
X = X.to(device)  # データをGPUに送る

scores = model(X)  # 順伝播 (直接modelを呼ぶ)
print(f'model output: {scores}')  # 出力 (batch_size, num_outputs)

pred_probab = nn.Softmax(dim=1)(scores)  # Softmax関数に通す (dim=確率の合計が1になる次元)
                                         # 評価時に確率を求めたい場合のみ。
                                         # 学習時はCrossEntropyLoss内のLogSoftmaxを利用する
                                         # 分類時はscores.argmaxを用い、Softmaxは省略するのが一般的(exp計算はコストがかかる)
print(f'Softmax output: {pred_probab}')

y_pred = pred_probab.argmax(dim=1)
print(f"予測結果: {y_pred}")

model output: tensor([[ 0.0553, -0.0663, -0.0259, -0.1072, -0.0782, -0.0080, -0.0303, -0.0943,
         -0.0626,  0.0061]], grad_fn=<AddmmBackward0>)
Softmax output: tensor([[0.1100, 0.0974, 0.1014, 0.0935, 0.0963, 0.1032, 0.1010, 0.0947, 0.0978,
         0.1047]], grad_fn=<SoftmaxBackward0>)
予測結果: tensor([0])


In [ ]:
# @title 損失関数と最適化手法の定義

loss_fn = nn.CrossEntropyLoss()  # 損失関数

# 最適化手法
learning_rate = 1e-3
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)  # 最適化するモデルのパラメータを渡す

In [ ]:
# @title torch内データセットの利用

from torchvision import models

# 利用可能なモデル一覧
print(f'モデル一覧: {models.list_models()}')

# モデルのみ読み込み
model = models.resnet50(weights=None)

# 事前学習済みモデルの読み込み
weights = models.ResNet50_Weights.DEFAULT
pre_trained_model = models.resnet50(weights=weights)  # 重みがディレクトリにダウンロードされる
preprocess = weights.transforms()  # 前処理
state_dict = weights.get_state_dict()
print(f'重みのバージョン: {weights.name}')
print(f'pthファイルのURL: {weights.url}')
print(f'分類カテゴリ: {weights.meta["categories"]}')
print(f'パラメータ数: {weights.meta["num_params"]}')

モデル一覧: ['alexnet', 'convnext_base', 'convnext_large', 'convnext_small', 'convnext_tiny', 'deeplabv3_mobilenet_v3_large', 'deeplabv3_resnet101', 'deeplabv3_resnet50', 'densenet121', 'densenet161', 'densenet169', 'densenet201', 'efficientnet_b0', 'efficientnet_b1', 'efficientnet_b2', 'efficientnet_b3', 'efficientnet_b4', 'efficientnet_b5', 'efficientnet_b6', 'efficientnet_b7', 'efficientnet_v2_l', 'efficientnet_v2_m', 'efficientnet_v2_s', 'fasterrcnn_mobilenet_v3_large_320_fpn', 'fasterrcnn_mobilenet_v3_large_fpn', 'fasterrcnn_resnet50_fpn', 'fasterrcnn_resnet50_fpn_v2', 'fcn_resnet101', 'fcn_resnet50', 'fcos_resnet50_fpn', 'googlenet', 'inception_v3', 'keypointrcnn_resnet50_fpn', 'lraspp_mobilenet_v3_large', 'maskrcnn_resnet50_fpn', 'maskrcnn_resnet50_fpn_v2', 'maxvit_t', 'mc3_18', 'mnasnet0_5', 'mnasnet0_75', 'mnasnet1_0', 'mnasnet1_3', 'mobilenet_v2', 'mobilenet_v3_large', 'mobilenet_v3_small', 'mvit_v1_b', 'mvit_v2_s', 'quantized_googlenet', 'quantized_inception_v3', 'quantized_mobil

---

## モデルの学習

In [ ]:
epochs = 10
print_freq = 100
for t in range(epochs):
  print(f"Epoch {t+1}")
  print('-' * 30)

  ## 学習ループ ##
  model.train()  # モデルを学習モードに (Dropoutとバッチ正規化を有効化)

  size = len(train_dataloader.dataset)

  for batch, (X, y) in enumerate(train_dataloader):
    X, y = X.to(device), y.to(device)  # データをGPUに送る
    optimizer.zero_grad()  # 重みの変数内の勾配値はbackwardの度に加算されるため、0に初期化

    # 順伝播
    pred = model(X)
    loss = loss_fn(pred, y)

    # 逆伝播
    loss.backward()  # 勾配を計算し、各重みの変数内に保存
    optimizer.step()  # 保存された勾配値を用いて重みの値を更新

    # 損失の表示
    if batch % print_freq == 0:
      loss, current = loss.item(), (batch + 1) * len(X)
      print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")


  ## 評価ループ ##
  model.eval()  # モデルを評価モードに (Dropoutとバッチ正規化を無効化)

  size = len(test_dataloader.dataset)
  num_batches = len(test_dataloader)
  test_loss, correct = 0, 0

  with torch.no_grad():  # 勾配の計算を行わない
    for X, y in test_dataloader:
      X, y = X.to(device), y.to(device)  # データをGPUに送る
      pred = model(X)
      test_loss += loss_fn(pred, y).item()
      correct += (pred.argmax(dim=1) == y).type(torch.float).sum().item()

  test_loss /= num_batches
  correct /= size
  print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")


---

## モデルの保存
- 拡張子は ".pth" or ".pt"
- CPUで保存したモデルはCPUで読み込む。GPUの場合も同じ


In [ ]:
# モデルの保存
torch.save(model.state_dict(), 'model_weights.pth')  # 重みのみ (推奨)

torch.save(model, 'model.pth')  # 重み + モデル構造(クラスファイルへのパスのみ保存)


# モデルのロード
model = NeuralNetwork()
model = model.to(device)  # GPU上で保存したため、GPU上で読み込み
model.load_state_dict(torch.load('model_weights.pth', weights_only=True))  # 重みのみ

model = torch.load('model.pth', weights_only=False)  # 重み + モデル構造

model.eval()  # モデルを評価モードに



# チェックポイントを保存・ロード
torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': loss
            }, "checkpoint.pth")

model = NeuralNetwork()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
checkpoint = torch.load("checkpoint.pth", weights_only=True)
model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
epoch = checkpoint['epoch']
loss = checkpoint['loss']

---

In [ ]:
# @title モデルを使用する

classes = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
x, y = test_data[0][0], test_data[0][1]

with torch.no_grad():  # 勾配の計算を行わない
    x = x.to(device)  # データをGPUに送る
    pred = model(x)
    predicted, actual = classes[pred[0].argmax(0)], classes[y]
    print(f'Predicted: "{predicted}", Actual: "{actual}"')


---

## PyTorchの基本要素

## Tensor
"Tensor" = PyTorchで使われる配列の型
- numpyのndarrayのように使える
- GPU上で動作する
- 自動で微分が行われる


In [ ]:
# Tensorの作成
tensor = torch.tensor([[1, 2], [3, 4]])  # 配列から
# shapeを指定して作成
tensor = torch.zeros((2, 2))  # 全て0
tensor = torch.ones((2, 2))  # 全て1
tensor = torch.rand((2, 2))  # [0, 1]の乱数

# numpyのndarrayとの相互変換 (device=cpuの時、オブジェクトの変更がもう片方に波及する)
ndarray = tensor.numpy()  # Tensor -> ndarray
tensor = torch.from_numpy(ndarray)  # ndarray -> Tensor

torch.tensor(1).item()  # Tensor -> int

# 属性
print(tensor.shape)  # 形状
print(tensor.dtype)  # 型
print(tensor.device)  # cpu, cuda,

# 計算
tensor.T  # 転置
tensor @ tensor.T  # 行列積
torch.cat([tensor, tensor, tensor], dim=1)  # 連結

torch.Size([2, 2])
torch.float32
cpu


tensor([[0.7153, 0.0330, 0.7153, 0.0330, 0.7153, 0.0330],
        [0.7255, 0.0099, 0.7255, 0.0099, 0.7255, 0.0099]])

In [ ]:
# @title 自動微分

# Tensorの計算では、計算手順が変数内に記録される
# 同時に逆伝播用の関数grad_fnが変数内に作成される

# 入力
x1 = torch.tensor(0.0)
x2 = torch.tensor(1.0)
print(f'入力: [{x1}, {x2}]')

# パラメータ
w1 = torch.rand((), requires_grad=True)  # requires_grad: 勾配を計算する必要がある変数
w2 = torch.rand((), requires_grad=True)
b = torch.rand((), requires_grad=True)
print(f'パラメータ: [{w1}, {w2}, {b}]')

# 順伝播
A = x1*w1 + x2*w2 + b
y = torch.nn.functional.relu(A)
print(f'出力: [{y}]')

# 損失の計算
y_hat = torch.tensor(1.0)
loss = torch.nn.functional.mse_loss(y, y_hat)
print(f'損失: {loss}')  # lossはこれまでの計算手順を保持している


# 勾配の計算
loss.backward()  # lossの計算手順の中で、requires_grad=Trueの変数を探し、lossをその変数で微分した勾配を変数.gradに格納する
                 # grad_fnを元に、微分の連鎖率を用いて計算される
                 # backwardを行うたびにgradの値は加算される -> 初期化が必要
print(f'grad: [{w1.grad}, {w2.grad}, {b.grad}]')  # gradの値を参照

# 勾配計算が無効になるブロック (パラメータ凍結用)
with torch.no_grad():
  pass


入力: [0.0, 1.0]
パラメータ: [0.23262709379196167, 0.6412057876586914, 0.37111276388168335]
出力: [1.0123186111450195]
損失: 0.00015174818690866232
grad: [0.0, 0.024637222290039062, 0.024637222290039062]


---

## その他

In [ ]:
# @title Ray Tuneを用いたハイパーパラメータチューニング
# https://pytorch.org/tutorials/beginner/hyperparameter_tuning_tutorial.html